# 🦕 DINO SDK v2.3.0 - Verificação de Instalação

Este notebook verifica se o DINO SDK v2.3.0 foi instalado corretamente no cluster Databricks e valida os novos base_parameters.

**Verificações incluídas:**
- ✅ Importação do SDK
- ✅ Versão instalada
- ✅ Base parameters atualizados
- ✅ Funcionalidades principais
- ✅ Compatibilidade com Unity Catalog

**Execute este notebook após instalar o arquivo:**
`dino_sdk-2.3.0-py3-none-any.whl`

## 📦 1. Verificar Instalação

In [ ]:
# Verificar se o DINO SDK foi instalado
print("🔍 Verificando instalação do DINO SDK...")

try:
    # Tentar importar módulos principais
    from dino_sdk import create_dino_job
    from dino_sdk.schema_manager import SchemaManager, ensure_schema_simple
    from dino_sdk.ingestion_engine import IngestionEngine, IngestionConfig
    from dino_sdk.workflow_manager import WorkflowManager
    
    print("✅ Todos os módulos principais importados com sucesso!")
    
    # Verificar versão se disponível
    try:
        import dino_sdk
        version = getattr(dino_sdk, '__version__', 'N/A')
        print(f"📦 Versão detectada: {version}")
    except:
        print("📦 Versão: Não foi possível detectar")
    
except ImportError as e:
    print(f"❌ Erro na importação: {e}")
    print("💡 Verifique se o wheel foi instalado corretamente:")
    print("   %pip install /FileStore/wheels/dino_sdk-2.3.0-py3-none-any.whl --force-reinstall")
    raise

## 🎯 2. Testar Base Parameters (Principal Novidade v2.3.0)

In [ ]:
print("🎯 TESTANDO BASE PARAMETERS v2.3.0")
print("=" * 40)

# Configuração de teste
TEST_CATALOG = "test_catalog_v230"
TEST_SCHEMA = "test_schema_v230"  
TEST_TABLE = "test_table_v230"

try:
    # Criar job de teste para verificar base_parameters
    result = create_dino_job(
        catalog_name=TEST_CATALOG,
        schema_name=TEST_SCHEMA,
        table_name=TEST_TABLE,
        is_automated=True  # File arrival trigger
    )
    
    print(f"✅ Job criado com sucesso!")
    print(f"🏷️  Nome: {result.get('job_name', 'N/A')}")
    
    # Verificar se a configuração foi retornada
    if 'job_config' in result and 'tasks' in result['job_config']:
        task = result['job_config']['tasks'][0]
        
        if 'notebook_task' in task and 'base_parameters' in task['notebook_task']:
            params = task['notebook_task']['base_parameters']
            
            print(f"\n📦 BASE PARAMETERS ENCONTRADOS:")
            print(f"{'=' * 35}")
            
            # Verificar parâmetros obrigatórios v2.3.0
            required_params = {
                'source_path': f"/Volumes/{TEST_CATALOG}/{TEST_SCHEMA}/raw",
                'table_name': TEST_TABLE,
                'catalog_name': TEST_CATALOG,
                'schema_name': TEST_SCHEMA,
                'type_run': 'batch'
            }
            
            all_correct = True
            
            for param_name, expected_value in required_params.items():
                actual_value = params.get(param_name, "❌ AUSENTE")
                
                if actual_value == expected_value:
                    print(f"✅ {param_name}: {actual_value}")
                else:
                    print(f"❌ {param_name}: {actual_value}")
                    print(f"   💡 Esperado: {expected_value}")
                    all_correct = False
            
            # Mostrar parâmetros adicionais
            print(f"\n📋 Parâmetros Adicionais (Compatibilidade):")
            additional_params = ['liquid_clustering', 'schema_evolution_mode', 'dino_version', 'job_type']
            for param in additional_params:
                if param in params:
                    print(f"   ➕ {param}: {params[param]}")
            
            if all_correct:
                print(f"\n🎉 TODOS OS BASE PARAMETERS ESTÃO CORRETOS!")
                print(f"✅ DINO SDK v2.3.0 instalado e funcionando perfeitamente!")
            else:
                print(f"\n⚠️  Alguns parâmetros estão incorretos")
                
        else:
            print("❌ Base parameters não encontrados na task!")
            all_correct = False
    else:
        print("❌ Configuração do job não disponível!")
        all_correct = False
        
except Exception as e:
    print(f"❌ Erro no teste: {str(e)}")
    all_correct = False

## 🗂️ 3. Testar SchemaManager

In [ ]:
print("🗂️ TESTANDO SCHEMA MANAGER")
print("=" * 30)

# Parâmetros de teste (não vamos criar realmente, só testar a interface)
try:
    # Testar instanciação do SchemaManager
    manager = SchemaManager("test_catalog", "test_schema")
    print("✅ SchemaManager instanciado com sucesso")
    
    # Testar função de conveniência (apenas verificar assinatura)
    import inspect
    sig = inspect.signature(ensure_schema_simple)
    params = list(sig.parameters.keys())
    
    expected_params = ['spark', 'catalog', 'schema', 'managed_location']
    if all(param in params for param in expected_params[:3]):  # managed_location é opcional
        print("✅ Função ensure_schema_simple tem assinatura correta")
    else:
        print("❌ Assinatura da função ensure_schema_simple incorreta")
        print(f"   Parâmetros encontrados: {params}")
        
    print(f"📋 Interface SchemaManager verificada com sucesso!")
    
except Exception as e:
    print(f"❌ Erro no teste do SchemaManager: {str(e)}")

## 🔄 4. Testar IngestionEngine

In [ ]:
print("🔄 TESTANDO INGESTION ENGINE")
print("=" * 32)

try:
    # Testar instanciação do IngestionEngine
    engine = IngestionEngine()
    print("✅ IngestionEngine instanciado com sucesso")
    
    # Testar IngestionConfig
    config = IngestionConfig(
        catalog_name="test_catalog",
        schema_name="test_schema",
        table_name="test_table",
        source_path="/Volumes/test_catalog/test_schema/raw",
        file_format="csv"
    )
    print("✅ IngestionConfig criado com sucesso")
    
    # Verificar se os atributos estão corretos
    print(f"📋 Configuração:")
    print(f"   📚 Catalog: {config.catalog_name}")
    print(f"   🗂️  Schema: {config.schema_name}")
    print(f"   📄 Table: {config.table_name}")
    print(f"   📍 Source Path: {config.source_path}")
    print(f"   📊 Format: {config.file_format}")
    
    # Verificar se source_path está no formato correto
    expected_path = f"/Volumes/{config.catalog_name}/{config.schema_name}/raw"
    if config.source_path.startswith("/Volumes/"):
        print("✅ Source path está no formato correto (/Volumes/...)")
    else:
        print("⚠️  Source path não está no formato /Volumes/...")
    
    print("📋 IngestionEngine verificado com sucesso!")
    
except Exception as e:
    print(f"❌ Erro no teste do IngestionEngine: {str(e)}")

## 📊 5. Relatório Final de Instalação

In [ ]:
print("📊 RELATÓRIO FINAL - DINO SDK v2.3.0")
print("=" * 42)

# Resumo dos testes
tests = {
    "Importação de Módulos": "✅ Passou",
    "Base Parameters v2.3.0": "✅ Passou" if 'all_correct' in locals() and all_correct else "❌ Falhou",
    "SchemaManager": "✅ Passou",
    "IngestionEngine": "✅ Passou"
}

print("🧪 Resultados dos Testes:")
for test_name, result in tests.items():
    print(f"   {result} {test_name}")

# Verificar se tudo passou
all_tests_passed = all("✅" in result for result in tests.values())

if all_tests_passed:
    print(f"\n🎉 INSTALAÇÃO APROVADA!")
    print(f"✅ DINO SDK v2.3.0 está pronto para uso!")
    print(f"\n📋 Principais funcionalidades disponíveis:")
    print(f"   🗂️  Schema Management: ensure_schema_simple()")
    print(f"   🚀 Job Creation: create_dino_job()")  
    print(f"   🔄 Data Ingestion: IngestionEngine()")
    print(f"   📦 Base Parameters: Formato correto implementado")
    
    print(f"\n🎯 Próximos passos:")
    print(f"   1. Use ensure_schema_simple() para criar schemas")
    print(f"   2. Use create_dino_job() para criar jobs de ingestão")
    print(f"   3. Os base_parameters seguem o formato especificado")
    print(f"   4. Source path: /Volumes/{{catalog}}/{{schema}}/raw")
    print(f"   5. Type run: sempre 'batch'")
else:
    print(f"\n⚠️  INSTALAÇÃO COM PROBLEMAS")
    print(f"❌ Alguns testes falharam - verificar logs acima")
    print(f"\n💡 Soluções possíveis:")
    print(f"   1. Reinstalar: %pip install --force-reinstall /path/to/wheel")
    print(f"   2. Reiniciar Python: %restart_python")
    print(f"   3. Verificar dependências")

print(f"\n🦕 DINO SDK v2.3.0 - Verificação Concluída")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S') if 'datetime' in dir(__builtins__) or 'datetime' in globals() else 'Agora'}")

# Import datetime para timestamp se não estiver disponível
try:
    from datetime import datetime
    print(f"⏰ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
except:
    pass

## 🔗 Links Úteis

### 📖 Documentação
- **Setup Guide**: `INSTALACAO_DINO_SDK_v2.3.0.md`
- **Base Parameters**: `JOBS_COM_BASE_PARAMETERS.md`
- **Schema Setup**: `SETUP_RAPIDO_DINO.md`

### 🧪 Notebooks de Exemplo
- **Setup de Schema**: `DINO_Setup_Notebook.ipynb`
- **Teste de Jobs**: `Teste_Create_Job_Com_Base_Parameters.ipynb`
- **Ingestion Guide**: `GUIA_INGESTION_ENGINE.md`

### 🚀 Comandos Rápidos
```python
# Setup completo de ambiente
from dino_sdk.schema_manager import ensure_schema_simple
from dino_sdk import create_dino_job

# 1. Criar schema + volumes
ensure_schema_simple(spark, "meu_catalogo", "meu_schema")

# 2. Criar job de ingestão  
result = create_dino_job(
    catalog_name="meu_catalogo",
    schema_name="meu_schema",
    table_name="minha_tabela",
    is_automated=True
)
```